# EXP21D-v7 — EXP 2B Sampled Neighborhood Audit
Label-independent paired audit of Uniform, Topology, and Importance sampling at fan-out `[15, 10]`. Labels are joined only after edges have been selected. This notebook was prepared locally and is not pushed automatically.


In [ ]:
%%writefile requirements.txt
numpy>=1.26
pandas>=2.1
matplotlib>=3.8
PyYAML>=6.0


In [ ]:
%%writefile config.kaggle.yaml
experiment:
  id: "EXP21D-v7-SamplerAudit-2B"
  seed: 42
data:
  transactions: "/kaggle/input/credit-card-transactions/credit_card_transactions-ibm_v2.csv"
  start_year: 2016
  max_rows: null
time_window: {reference_date: "2018-01-01", train_days: 300, val_days: 30, gap_days: 8, test_days: 60}
sampling:
  methods: ["uniform", "topology", "importance"]
  num_neighbors: [15, 10]
  smoothing_alpha: 0.1
audit:
  splits: ["test"]
  population_roots: 10000
  max_fraud_roots: null
  repetitions: 5
  save_edge_level: true
  figure_sample_edges: 100000
output:
  result_dir: "/kaggle/working/result/exp21d_v7_sampler_audit_2b"
  snapshot_source: true


In [ ]:
%%writefile config.py
from pathlib import Path

import yaml


def load_config(path: str = "config.reference.yaml") -> dict:
    source = Path(path)
    if not source.exists():
        raise FileNotFoundError(f"Konfigurasi tidak ditemukan: {source}")
    config = yaml.safe_load(source.read_text(encoding="utf-8"))
    if config["sampling"]["num_neighbors"] != [15, 10]:
        raise ValueError("EXP 2B mengunci fan-out [15, 10]")
    if config["sampling"]["methods"] != ["uniform", "topology", "importance"]:
        raise ValueError("EXP 2B memerlukan urutan Uniform, Topology, Importance")
    if not 0 < float(config["sampling"]["smoothing_alpha"]) <= 1:
        raise ValueError("smoothing_alpha harus berada pada (0, 1]")
    return config


In [ ]:
%%writefile artifacts.py
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


def write_json(path: str | Path, value: Any) -> None:
    target = Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_suffix(target.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str) + "\n", encoding="utf-8")
    temporary.replace(target)


def append_event(path: str | Path, event: str, **payload: Any) -> None:
    record = {"timestamp_utc": datetime.now(timezone.utc).isoformat(), "event": event, **payload}
    with Path(path).open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")


def artifact_index(root: str | Path) -> dict[str, Any]:
    base = Path(root)
    files = [
        {"path": item.relative_to(base).as_posix(), "size_bytes": item.stat().st_size}
        for item in sorted(base.rglob("*"))
        if item.is_file() and item.name != "artifact_index.json"
    ]
    result = {"file_count": len(files), "files": files}
    write_json(base / "artifact_index.json", result)
    return result


In [ ]:
%%writefile data_loader.py
from __future__ import annotations

from datetime import timedelta

import numpy as np
import pandas as pd


COLUMNS = ["User", "Merchant Name", "Is Fraud?", "Year", "Month", "Day", "Time"]


def load_temporal_splits(
    path: str,
    start_year: int,
    max_rows: int | None,
    reference_date: str,
    train_days: int,
    val_days: int,
    gap_days: int,
    test_days: int,
) -> tuple[dict[str, pd.DataFrame], dict]:
    start = pd.Timestamp(reference_date)
    horizon_end = start + timedelta(days=train_days + val_days + gap_days + test_days)
    chunks, collected = [], 0
    for chunk in pd.read_csv(path, usecols=COLUMNS, chunksize=1_000_000):
        lower_year = max(start_year, start.year)
        chunk = chunk.loc[(chunk["Year"] >= lower_year) & (chunk["Year"] <= horizon_end.year)].copy()
        if max_rows is not None:
            chunk = chunk.iloc[: max(0, max_rows - collected)]
        if len(chunk):
            chunk["source_row"] = chunk.index.to_numpy(dtype=np.int64)
            chunks.append(chunk)
            collected += len(chunk)
        if max_rows is not None and collected >= max_rows:
            break
    if not chunks:
        raise ValueError("Tidak ada baris setelah filter tahun")
    frame = pd.concat(chunks, ignore_index=True)
    frame["Time"] = frame["Time"].fillna("00:00")
    frame["datetime"] = pd.to_datetime(
        frame["Year"].astype(str) + "-" + frame["Month"].astype(str).str.zfill(2) + "-"
        + frame["Day"].astype(str).str.zfill(2) + " " + frame["Time"]
    )
    frame["label"] = frame["Is Fraud?"].map({"Yes": 1, "No": 0})
    if frame["label"].isna().any():
        raise ValueError("Label selain Yes/No ditemukan")
    frame = frame.sort_values(["datetime", "source_row"], kind="stable").reset_index(drop=True)
    boundaries = {
        "train": (start, start + timedelta(days=train_days)),
        "validation": (start + timedelta(days=train_days), start + timedelta(days=train_days + val_days)),
        "gap": (start + timedelta(days=train_days + val_days), start + timedelta(days=train_days + val_days + gap_days)),
        "test": (
            start + timedelta(days=train_days + val_days + gap_days),
            start + timedelta(days=train_days + val_days + gap_days + test_days),
        ),
    }
    splits = {}
    statistics = {"boundaries": {}, "splits": {}}
    for name, (lower, upper) in boundaries.items():
        split = frame.loc[(frame["datetime"] >= lower) & (frame["datetime"] < upper), ["User", "Merchant Name", "label", "datetime", "source_row"]].copy()
        split = split.reset_index(drop=True)
        split["transaction_id"] = np.arange(len(split), dtype=np.int32)
        splits[name] = split
        statistics["boundaries"][name] = [str(lower), str(upper)]
        statistics["splits"][name] = {
            "rows": int(len(split)), "fraud": int(split["label"].sum()),
            "fraud_rate": float(split["label"].mean()) if len(split) else None,
            "users": int(split["User"].nunique()), "merchants": int(split["Merchant Name"].nunique()),
        }
    return splits, statistics


In [ ]:
%%writefile sampling_core.py
from __future__ import annotations

import numpy as np


METHODS = ("uniform", "topology", "importance")
CHANNELS = ("via_user", "via_merchant")


def minmax_plus_floor(values: np.ndarray, alpha: float) -> np.ndarray:
    values = np.asarray(values, dtype=np.float64)
    if not 0 < alpha <= 1:
        raise ValueError("alpha harus berada pada (0, 1]")
    if values.size == 0 or np.isclose(values.max(), values.min()):
        return np.ones_like(values, dtype=np.float64)
    return (values - values.min()) / (values.max() - values.min() + 1e-8) + alpha


def candidate_score(counterpart_degree: np.ndarray, method: str, alpha: float = 0.1) -> np.ndarray:
    degree = np.maximum(np.asarray(counterpart_degree, dtype=np.float64), 1.0)
    if method == "uniform":
        return np.ones_like(degree)
    if method == "topology":
        raw = 1.0 / np.log1p(degree)
    elif method == "importance":
        raw = np.log1p(degree)
    else:
        raise ValueError(f"Metode tidak didukung: {method}")
    return minmax_plus_floor(raw, alpha)


def encode_entities(values) -> tuple[np.ndarray, np.ndarray]:
    codes, uniques = values.factorize(sort=False)
    if (codes < 0).any():
        raise ValueError("Entity ID kosong tidak diperbolehkan")
    return codes.astype(np.int32, copy=False), np.asarray(uniques)


def build_csr(entity_codes: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    number = int(entity_codes.max()) + 1 if len(entity_codes) else 0
    degree = np.bincount(entity_codes, minlength=number).astype(np.int32)
    order = np.argsort(entity_codes, kind="stable").astype(np.int32)
    offsets = np.empty(number + 1, dtype=np.int64)
    offsets[0] = 0
    np.cumsum(degree, out=offsets[1:])
    return order, offsets, degree


def entity_candidates(order: np.ndarray, offsets: np.ndarray, entity: int) -> np.ndarray:
    return order[offsets[entity] : offsets[entity + 1]]


def paired_rng(seed: int, repetition: int, root: int, channel: int) -> np.random.Generator:
    return np.random.default_rng(np.random.SeedSequence([seed, repetition, root, channel]))


def sample_without_replacement(
    candidates: np.ndarray,
    global_weights: np.ndarray,
    take: int,
    rng: np.random.Generator,
) -> np.ndarray:
    count = len(candidates)
    if count <= take:
        return candidates.copy()
    local = global_weights[candidates].astype(np.float64, copy=False)
    probability = local / local.sum()
    return rng.choice(candidates, size=take, replace=False, p=probability).astype(np.int32, copy=False)


def jaccard(left: np.ndarray, right: np.ndarray) -> float:
    a, b = set(map(int, left)), set(map(int, right))
    union = a | b
    return float(len(a & b) / len(union)) if union else 1.0


def distribution(values: np.ndarray) -> dict[str, float | int | None]:
    values = np.asarray(values)
    if values.size == 0:
        return {
            "count": 0, "mean": None, "std": None, "min": None,
            "q25": None, "median": None, "q75": None,
            "p90": None, "p95": None, "p99": None, "max": None,
        }
    q25, median, q75, p90, p95, p99 = np.quantile(values, [0.25, 0.5, 0.75, 0.90, 0.95, 0.99])
    return {
        "count": int(values.size), "mean": float(values.mean()), "std": float(values.std()),
        "min": float(values.min()), "q25": float(q25), "median": float(median),
        "q75": float(q75), "p90": float(p90), "p95": float(p95),
        "p99": float(p99), "max": float(values.max()),
    }


In [ ]:
%%writefile visualize_results.py
from __future__ import annotations

import argparse
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


COLORS = {"uniform": "#4472C4", "topology": "#ED7D31", "importance": "#70AD47"}


def save(fig, output: Path, stem: str) -> list[str]:
    names = []
    for suffix in ("png", "pdf"):
        target = output / f"{stem}.{suffix}"
        fig.savefig(target, dpi=300, bbox_inches="tight")
        names.append(target.name)
    plt.close(fig)
    return names


def create_visualizations(split_dir: str | Path, sample_edges: int = 100_000, seed: int = 42) -> None:
    root = Path(split_dir)
    figures = root / "figures"
    figures.mkdir(exist_ok=True)
    degree = pd.read_csv(root / "degree_distribution.csv")
    homophily = pd.read_csv(root / "effective_homophily.csv")
    overlap = pd.read_csv(root / "jaccard_overlap.csv")
    generated = []

    population = degree[(degree["cohort"] == "population") & (degree["hop"] == 2)].copy()
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=False)
    for axis, channel in zip(axes, ("via_user", "via_merchant")):
        subset = population[population["channel"] == channel].set_index("method").reindex(COLORS)
        x = np.arange(len(subset))
        median = subset["median"].to_numpy()
        lower = median - subset["q25"].to_numpy()
        upper = subset["q75"].to_numpy() - median
        axis.bar(x, median, color=[COLORS[item] for item in subset.index], alpha=0.85)
        axis.errorbar(x, median, yerr=np.vstack([lower, upper]), fmt="none", color="black", capsize=4)
        axis.set_xticks(x, [item.title() for item in subset.index], rotation=15)
        axis.set_title(channel.replace("_", " ").title())
        axis.set_ylabel("Selected counterpart degree (median, IQR)")
        axis.grid(axis="y", alpha=0.25)
    fig.suptitle("EXP 2B — Degree distribution of sampled second-hop transactions")
    generated += save(fig, figures, "fig1_selected_counterpart_degree")

    selected_overlap = overlap[(overlap["cohort"] == "population") & (overlap["channel"] == "combined")].copy()
    labels = [f"{left.title()}–{right.title()}" for left, right in zip(selected_overlap["left"], selected_overlap["right"])]
    fig, axis = plt.subplots(figsize=(7.5, 4.5))
    axis.bar(labels, selected_overlap["median"], color="#5B9BD5")
    axis.errorbar(
        np.arange(len(labels)), selected_overlap["median"],
        yerr=np.vstack([selected_overlap["median"] - selected_overlap["q25"], selected_overlap["q75"] - selected_overlap["median"]]),
        fmt="none", color="black", capsize=4,
    )
    axis.set_ylim(0, 1)
    axis.set_ylabel("Root-local Jaccard overlap")
    axis.set_title("Combined user/merchant sampled-edge overlap")
    axis.grid(axis="y", alpha=0.25)
    generated += save(fig, figures, "fig2_pairwise_jaccard_overlap")

    fraud = homophily[homophily["cohort"] == "fraud_roots"].copy()
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    for axis, channel in zip(axes, ("via_user", "via_merchant")):
        subset = fraud[fraud["channel"] == channel].set_index("method").reindex(COLORS)
        axis.bar(np.arange(len(subset)), subset["fraud_lift_over_split"], color=[COLORS[item] for item in subset.index])
        axis.set_xticks(np.arange(len(subset)), [item.title() for item in subset.index], rotation=15)
        axis.set_title(channel.replace("_", " ").title())
        axis.set_ylabel("Non-self neighbor fraud lift")
        axis.axhline(1.0, color="black", linestyle="--", linewidth=1)
        axis.grid(axis="y", alpha=0.25)
    fig.suptitle("Effective fraud signal around fraud roots (post-hoc labels)")
    generated += save(fig, figures, "fig3_effective_fraud_signal")

    edge_data = np.load(root / "selected_edges.npz")
    rng = np.random.default_rng(seed)
    sampled_rows = []
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
    for channel_index, (axis, channel) in enumerate(zip(axes, ("via_user", "via_merchant"))):
        for method_index, method in enumerate(COLORS):
            available = np.flatnonzero(
                (edge_data["method"] == method_index) & (edge_data["channel"] == channel_index) & edge_data["population_cohort"]
            )
            chosen = rng.choice(available, size=min(sample_edges, len(available)), replace=False)
            values = np.sort(edge_data["counterpart_degree"][chosen].astype(np.float64))
            probability = np.arange(1, len(values) + 1) / len(values)
            axis.step(values, probability, where="post", label=method.title(), color=COLORS[method])
            stride = max(1, len(values) // 1000)
            sampled_rows.extend(
                {"channel": channel, "method": method, "degree": float(value), "ecdf": float(ecdf)}
                for value, ecdf in zip(values[::stride], probability[::stride])
            )
        axis.set_xscale("log")
        axis.set_xlabel("Counterpart degree (log scale)")
        axis.set_ylabel("Empirical CDF")
        axis.set_title(channel.replace("_", " ").title())
        axis.grid(alpha=0.25)
        axis.legend(frameon=False)
    fig.suptitle("Distribution shift induced by each sampler")
    generated += save(fig, figures, "fig4_counterpart_degree_ecdf")
    edge_data.close()
    pd.DataFrame(sampled_rows).to_csv(figures / "data_counterpart_degree_ecdf.csv", index=False)
    degree.to_csv(figures / "data_degree_distribution.csv", index=False)
    overlap.to_csv(figures / "data_jaccard_overlap.csv", index=False)
    homophily.to_csv(figures / "data_effective_homophily.csv", index=False)
    manifest = {
        "figures": generated,
        "source_data": [
            "data_counterpart_degree_ecdf.csv", "data_degree_distribution.csv",
            "data_jaccard_overlap.csv", "data_effective_homophily.csv",
        ],
    }
    (figures / "figures_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--split-dir", required=True)
    parser.add_argument("--sample-edges", type=int, default=100_000)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()
    create_visualizations(args.split_dir, args.sample_edges, args.seed)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile main.py
from __future__ import annotations

import argparse
import hashlib
import json
import platform
import shutil
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

from artifacts import append_event, artifact_index, write_json
from config import load_config
from data_loader import load_temporal_splits
from sampling_core import (
    CHANNELS,
    METHODS,
    build_csr,
    candidate_score,
    distribution,
    encode_entities,
    entity_candidates,
    jaccard,
    paired_rng,
    sample_without_replacement,
)
from visualize_results import create_visualizations


PAIR_NAMES = (("uniform", "topology"), ("uniform", "importance"), ("topology", "importance"))
JACCARD_CHANNELS = ("via_user", "via_merchant", "combined")


def stable_hash(value: object, length: int = 12) -> str:
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str).encode()).hexdigest()[:length]


def resolve_path(value: str, config_path: Path) -> Path:
    path = Path(value)
    return path if path.is_absolute() else (config_path.parent / path).resolve()


def resolve_dataset(config: dict, config_path: Path) -> Path:
    path = resolve_path(config["data"]["transactions"], config_path)
    if path.exists():
        return path
    kaggle = Path("/kaggle/input")
    if kaggle.exists():
        matches = sorted(kaggle.rglob("credit_card_transactions-ibm_v2.csv"))
        if matches:
            return matches[0]
    raise FileNotFoundError(f"Dataset tidak ditemukan: {path}")


def choose_roots(labels: np.ndarray, population_count: int, max_fraud: int | None, seed: int) -> dict[str, np.ndarray]:
    rng = np.random.default_rng(seed)
    population = np.sort(rng.choice(len(labels), size=min(population_count, len(labels)), replace=False).astype(np.int32))
    fraud = np.flatnonzero(labels == 1).astype(np.int32)
    if max_fraud is not None and len(fraud) > max_fraud:
        fraud = np.sort(rng.choice(fraud, size=max_fraud, replace=False).astype(np.int32))
    roots = np.union1d(population, fraud).astype(np.int32)
    return {"all": roots, "population": population, "fraud": fraud}


def allocate_edges(size: int) -> dict[str, np.ndarray]:
    return {
        "method": np.empty(size, dtype=np.uint8), "repetition": np.empty(size, dtype=np.uint8),
        "channel": np.empty(size, dtype=np.uint8), "root_tx": np.empty(size, dtype=np.int32),
        "candidate_tx": np.empty(size, dtype=np.int32), "root_label": np.empty(size, dtype=np.uint8),
        "candidate_label": np.empty(size, dtype=np.uint8), "shared_entity_degree": np.empty(size, dtype=np.int32),
        "counterpart_degree": np.empty(size, dtype=np.int32), "sampling_weight": np.empty(size, dtype=np.float32),
        "is_self": np.empty(size, dtype=np.bool_), "population_cohort": np.empty(size, dtype=np.bool_),
        "fraud_cohort": np.empty(size, dtype=np.bool_),
    }


def allocate_jaccard(size: int) -> dict[str, np.ndarray]:
    return {
        "pair": np.empty(size, dtype=np.uint8), "repetition": np.empty(size, dtype=np.uint8),
        "channel": np.empty(size, dtype=np.uint8), "root_tx": np.empty(size, dtype=np.int32),
        "value": np.empty(size, dtype=np.float32), "exact_match": np.empty(size, dtype=np.bool_),
        "population_cohort": np.empty(size, dtype=np.bool_), "fraud_cohort": np.empty(size, dtype=np.bool_),
    }


def trim(arrays: dict[str, np.ndarray], count: int) -> dict[str, np.ndarray]:
    return {name: values[:count] for name, values in arrays.items()}


def audit_split(frame: pd.DataFrame, config: dict, output: Path, split: str, seed: int) -> dict:
    started = time.time()
    labels = frame["label"].to_numpy(dtype=np.uint8)
    transaction_ids = frame["transaction_id"].to_numpy(dtype=np.int32)
    source_rows = frame["source_row"].to_numpy(dtype=np.int64)
    user_code, user_values = encode_entities(frame["User"])
    merchant_code, merchant_values = encode_entities(frame["Merchant Name"])
    user_order, user_offsets, user_degree = build_csr(user_code)
    merchant_order, merchant_offsets, merchant_degree = build_csr(merchant_code)
    user_degree_per_tx = user_degree[user_code]
    merchant_degree_per_tx = merchant_degree[merchant_code]

    audit_cfg, sampling_cfg = config["audit"], config["sampling"]
    k1, k2 = map(int, sampling_cfg["num_neighbors"])
    repetitions = int(audit_cfg["repetitions"])
    roots = choose_roots(labels, int(audit_cfg["population_roots"]), audit_cfg.get("max_fraud_roots"), seed)
    root_set, population_set, fraud_set = roots["all"], set(map(int, roots["population"])), set(map(int, roots["fraud"]))
    weights = {
        method: {
            "via_user": candidate_score(merchant_degree_per_tx, method, float(sampling_cfg["smoothing_alpha"])),
            "via_merchant": candidate_score(user_degree_per_tx, method, float(sampling_cfg["smoothing_alpha"])),
        }
        for method in METHODS
    }
    max_edges = len(METHODS) * repetitions * len(CHANNELS) * len(root_set) * k2
    max_jaccard = len(PAIR_NAMES) * repetitions * len(JACCARD_CHANNELS) * len(root_set)
    edges, local_overlap = allocate_edges(max_edges), allocate_jaccard(max_jaccard)
    edge_count = overlap_count = 0

    for repetition in range(repetitions):
        for root in root_set:
            root_int = int(root)
            candidates_by_channel = (
                entity_candidates(user_order, user_offsets, int(user_code[root_int])),
                entity_candidates(merchant_order, merchant_offsets, int(merchant_code[root_int])),
            )
            selections: dict[str, tuple[np.ndarray, np.ndarray]] = {}
            for method_index, method in enumerate(METHODS):
                method_selections = []
                for channel_index, (channel, candidates) in enumerate(zip(CHANNELS, candidates_by_channel)):
                    rng = paired_rng(seed, repetition, root_int, channel_index)
                    selected = sample_without_replacement(candidates, weights[method][channel], k2, rng)
                    method_selections.append(selected)
                    count = len(selected)
                    target = slice(edge_count, edge_count + count)
                    shared_degree = user_degree[user_code[root_int]] if channel == "via_user" else merchant_degree[merchant_code[root_int]]
                    counterpart = merchant_degree_per_tx[selected] if channel == "via_user" else user_degree_per_tx[selected]
                    edges["method"][target] = method_index
                    edges["repetition"][target] = repetition
                    edges["channel"][target] = channel_index
                    edges["root_tx"][target] = root_int
                    edges["candidate_tx"][target] = selected
                    edges["root_label"][target] = labels[root_int]
                    edges["candidate_label"][target] = labels[selected]
                    edges["shared_entity_degree"][target] = shared_degree
                    edges["counterpart_degree"][target] = counterpart
                    edges["sampling_weight"][target] = weights[method][channel][selected]
                    edges["is_self"][target] = selected == root_int
                    edges["population_cohort"][target] = root_int in population_set
                    edges["fraud_cohort"][target] = root_int in fraud_set
                    edge_count += count
                selections[method] = (method_selections[0], method_selections[1])
            for pair_index, (left, right) in enumerate(PAIR_NAMES):
                comparisons = (
                    (selections[left][0], selections[right][0]),
                    (selections[left][1], selections[right][1]),
                    (
                        np.concatenate([selections[left][0], selections[left][1] + len(labels)]),
                        np.concatenate([selections[right][0], selections[right][1] + len(labels)]),
                    ),
                )
                for channel_index, (left_values, right_values) in enumerate(comparisons):
                    value = jaccard(left_values, right_values)
                    local_overlap["pair"][overlap_count] = pair_index
                    local_overlap["repetition"][overlap_count] = repetition
                    local_overlap["channel"][overlap_count] = channel_index
                    local_overlap["root_tx"][overlap_count] = root_int
                    local_overlap["value"][overlap_count] = value
                    local_overlap["exact_match"][overlap_count] = np.isclose(value, 1.0)
                    local_overlap["population_cohort"][overlap_count] = root_int in population_set
                    local_overlap["fraud_cohort"][overlap_count] = root_int in fraud_set
                    overlap_count += 1

    edges, local_overlap = trim(edges, edge_count), trim(local_overlap, overlap_count)
    output.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        output / "root_cohorts.npz", all_roots=root_set, population_roots=roots["population"], fraud_roots=roots["fraud"],
        transaction_ids=transaction_ids, source_rows=source_rows, labels=labels,
        user_code=user_code, merchant_code=merchant_code, user_degree=user_degree, merchant_degree=merchant_degree,
    )
    if audit_cfg.get("save_edge_level", True):
        np.savez_compressed(output / "selected_edges.npz", **edges)
        np.savez_compressed(output / "local_jaccard.npz", **local_overlap)

    degree_rows, signal_rows, overlap_rows = [], [], []
    for method_index, method in enumerate(METHODS):
        for channel_index, channel in enumerate(CHANNELS):
            base = (edges["method"] == method_index) & (edges["channel"] == channel_index)
            for cohort, cohort_mask in (
                ("population", edges["population_cohort"]),
                ("fraud_roots", edges["fraud_cohort"] & (edges["root_label"] == 1)),
            ):
                mask = base & cohort_mask
                degree_rows.append({
                    "split": split, "hop": 2, "method": method, "channel": channel,
                    "cohort": cohort, "degree_semantics": "candidate_counterpart_entity_degree",
                    **distribution(edges["counterpart_degree"][mask]),
                })
                nonself = mask & ~edges["is_self"]
                candidate_rate = float(edges["candidate_label"][nonself].mean()) if nonself.any() else None
                same_label = float((edges["candidate_label"][nonself] == edges["root_label"][nonself]).mean()) if nonself.any() else None
                signal_rows.append({
                    "split": split, "method": method, "channel": channel, "cohort": cohort,
                    "selected_edges": int(mask.sum()), "nonself_edges": int(nonself.sum()),
                    "nonself_candidate_fraud_rate": candidate_rate,
                    "fraud_lift_over_split": None if candidate_rate is None else candidate_rate / max(float(labels.mean()), 1e-12),
                    "same_label_rate": same_label,
                    "self_edge_rate": float(edges["is_self"][mask].mean()) if mask.any() else None,
                })
    for method in METHODS:
        for channel, per_transaction_degree in (
            ("via_user", user_degree_per_tx),
            ("via_merchant", merchant_degree_per_tx),
        ):
            for cohort, cohort_roots in (
                ("population", roots["population"]),
                ("fraud_roots", roots["fraud"]),
            ):
                degree_rows.append({
                    "split": split, "hop": 1, "method": method, "channel": channel,
                    "cohort": cohort, "degree_semantics": "selected_entity_transaction_degree",
                    **distribution(per_transaction_degree[cohort_roots]),
                })
    for pair_index, (left, right) in enumerate(PAIR_NAMES):
        for channel_index, channel in enumerate(JACCARD_CHANNELS):
            base = (local_overlap["pair"] == pair_index) & (local_overlap["channel"] == channel_index)
            for cohort, cohort_mask in (("population", local_overlap["population_cohort"]), ("fraud_roots", local_overlap["fraud_cohort"])):
                mask = base & cohort_mask
                stats = distribution(local_overlap["value"][mask])
                overlap_rows.append({
                    "split": split, "left": left, "right": right, "channel": channel, "cohort": cohort,
                    **stats, "exact_match_rate": float(local_overlap["exact_match"][mask].mean()) if mask.any() else None,
                })

    degree_frame, signal_frame, overlap_frame = pd.DataFrame(degree_rows), pd.DataFrame(signal_rows), pd.DataFrame(overlap_rows)
    degree_frame.to_csv(output / "degree_distribution.csv", index=False)
    signal_frame.to_csv(output / "effective_homophily.csv", index=False)
    overlap_frame.to_csv(output / "jaccard_overlap.csv", index=False)
    population = roots["population"]
    hop1 = {
        "via_user_entity_degree": distribution(user_degree_per_tx[population]),
        "via_merchant_entity_degree": distribution(merchant_degree_per_tx[population]),
        "user_channel_no_competition_rate": float((user_degree_per_tx[population] <= k2).mean()),
        "merchant_channel_no_competition_rate": float((merchant_degree_per_tx[population] <= k2).mean()),
        "declared_hop1_fanout": k1,
        "effective_hop1_neighbors_per_relation": 1,
        "declared_hop2_fanout": k2,
    }
    summary = {
        "split": split, "seed": seed, "sampling_is_label_independent": True,
        "labels_used_only_posthoc": True, "rows": int(len(frame)), "fraud": int(labels.sum()),
        "fraud_rate": float(labels.mean()), "population_roots": int(len(roots["population"])),
        "fraud_roots": int(len(roots["fraud"])), "unique_audited_roots": int(len(root_set)),
        "repetitions": repetitions, "selected_edge_records": int(edge_count),
        "local_jaccard_records": int(overlap_count), "hop1": hop1,
        "method_encoding": {name: index for index, name in enumerate(METHODS)},
        "channel_encoding": {name: index for index, name in enumerate(CHANNELS)},
        "pair_encoding": {f"{left}__{right}": index for index, (left, right) in enumerate(PAIR_NAMES)},
        "runtime_seconds": time.time() - started,
    }
    write_json(output / "summary.json", summary)
    write_json(output / "edge_level_schema.json", {
        "selected_edges.npz": {name: str(values.dtype) for name, values in edges.items()},
        "local_jaccard.npz": {name: str(values.dtype) for name, values in local_overlap.items()},
        "root_cohorts.npz": {
            "derivation_note": "Full split labels, entity codes, and entity-degree lookup arrays allow additional selected-edge metrics and degree strata without rerunning sampling.",
            "transaction_axis_fields": ["transaction_ids", "source_rows", "labels", "user_code", "merchant_code"],
            "entity_lookup_fields": ["user_degree", "merchant_degree"],
        },
        "primary_homophily_metric": "nonself_candidate_fraud_rate",
        "edge_identity": ["repetition", "method", "channel", "root_tx", "candidate_tx"],
    })
    del user_values, merchant_values
    return summary


def write_report(run_dir: Path, summaries: list[dict]) -> None:
    lines = ["# EXP21D EXP 2B — Sampled Neighborhood Audit", "", "Sampling dilakukan tanpa menggunakan label; label dibaca hanya setelah edge terpilih untuk audit homofili.", ""]
    for item in summaries:
        lines += [
            f"## Split {item['split']}", "",
            f"- Rows: {item['rows']:,}; fraud: {item['fraud']:,}; audited population roots: {item['population_roots']:,}.",
            f"- Repetitions: {item['repetitions']}; selected edge records: {item['selected_edge_records']:,}.",
            f"- User entities with degree <= 10: {item['hop1']['user_channel_no_competition_rate']:.2%}.",
            f"- Merchant entities with degree <= 10: {item['hop1']['merchant_channel_no_competition_rate']:.2%}.", "",
        ]
    lines += ["CSV dan NPZ di setiap folder split menyimpan statistik agregat dan edge-level untuk analisis ulang.", ""]
    (run_dir / "audit_report.md").write_text("\n".join(lines), encoding="utf-8")


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="EXP 2B sampled-neighborhood audit")
    parser.add_argument("--config", default="config.reference.yaml")
    parser.add_argument("--seed", type=int)
    parser.add_argument("--splits", nargs="+", choices=["train", "validation", "test"])
    parser.add_argument("--population-roots", type=int)
    parser.add_argument("--repetitions", type=int)
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    is_smoke = args.population_roots is not None or args.repetitions is not None
    config_path = Path(args.config).resolve()
    config = load_config(str(config_path))
    if not config["audit"].get("save_edge_level", True):
        raise ValueError("EXP 2B mewajibkan edge-level output agar metrik dapat dihitung ulang")
    if args.population_roots is not None:
        config["audit"]["population_roots"] = args.population_roots
    if args.repetitions is not None:
        config["audit"]["repetitions"] = args.repetitions
    if args.splits:
        config["audit"]["splits"] = args.splits
    seed = int(args.seed if args.seed is not None else config["experiment"]["seed"])
    dataset = resolve_dataset(config, config_path)
    output_root = resolve_path(config["output"]["result_dir"], config_path)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run_id = f"exp21d_v7_2b_seed{seed}_{stable_hash(config)}_{stamp}"
    run_dir = output_root / "runs" / run_id
    run_dir.mkdir(parents=True, exist_ok=False)
    events = run_dir / "events.jsonl"
    manifest = {
        "experiment_id": config["experiment"]["id"], "run_id": run_id, "status": "running", "seed": seed,
        "scientific_status": "smoke_only_not_for_paper" if is_smoke else "paper_candidate_single_seed_audit",
        "created_at_utc": datetime.now(timezone.utc).isoformat(), "dataset": {"path": str(dataset), "size_bytes": dataset.stat().st_size},
        "python": sys.version, "platform": platform.platform(),
    }
    write_json(run_dir / "manifest.json", manifest)
    (run_dir / "resolved_config.yaml").write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
    append_event(events, "run_started", run_id=run_id, splits=config["audit"]["splits"])
    if config["output"].get("snapshot_source", True):
        snapshot = run_dir / "code_snapshot"
        snapshot.mkdir()
        for item in Path(__file__).resolve().parent.iterdir():
            if item.is_file() and item.suffix.lower() in {".py", ".yaml", ".md", ".txt"}:
                shutil.copy2(item, snapshot / item.name)
    window = config["time_window"]
    splits, statistics = load_temporal_splits(
        str(dataset), int(config["data"]["start_year"]), config["data"].get("max_rows"), window["reference_date"],
        int(window["train_days"]), int(window["val_days"]), int(window["gap_days"]), int(window["test_days"]),
    )
    write_json(run_dir / "window_statistics.json", statistics)
    summaries = []
    for index, split in enumerate(config["audit"]["splits"]):
        append_event(events, "split_audit_started", split=split)
        summary = audit_split(splits[split], config, run_dir / split, split, seed + index * 1000)
        summaries.append(summary)
        create_visualizations(run_dir / split, int(config["audit"]["figure_sample_edges"]), seed)
        append_event(events, "split_audit_complete", split=split, runtime_seconds=summary["runtime_seconds"])
    write_report(run_dir, summaries)
    manifest.update({"status": "complete", "completed_at_utc": datetime.now(timezone.utc).isoformat(), "splits": summaries})
    write_json(run_dir / "manifest.json", manifest)
    index = artifact_index(run_dir)
    append_event(events, "run_complete", artifact_count=index["file_count"])
    artifact_index(run_dir)
    print(run_dir)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile validate_audit.py
from __future__ import annotations

import inspect
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

from artifacts import write_json
from main import audit_split
from sampling_core import (
    METHODS,
    build_csr,
    candidate_score,
    entity_candidates,
    jaccard,
    paired_rng,
    sample_without_replacement,
)
from visualize_results import create_visualizations


def main() -> None:
    counterpart_degree = np.array([1, 1, 2, 5, 20, 100], dtype=np.int32)
    candidates = np.arange(len(counterpart_degree), dtype=np.int32)
    selected_means = {method: [] for method in METHODS}
    first_selection = {}
    for repetition in range(2000):
        for method in METHODS:
            weights = candidate_score(counterpart_degree, method, 0.1)
            selected = sample_without_replacement(candidates, weights, 2, paired_rng(42, repetition, 7, 0))
            selected_means[method].append(float(counterpart_degree[selected].mean()))
            if repetition == 0:
                first_selection[method] = selected
    mean = {method: float(np.mean(values)) for method, values in selected_means.items()}
    codes = np.array([0, 1, 0, 2, 1, 0], dtype=np.int32)
    order, offsets, degree = build_csr(codes)

    frame = pd.DataFrame({
        "User": [1, 1, 1, 1, 2, 2, 3, 3, 3, 4, 4, 4],
        "Merchant Name": [10, 11, 11, 12, 10, 12, 12, 12, 13, 10, 13, 14],
        "label": [0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1],
        "datetime": pd.date_range("2018-01-01", periods=12, freq="h"),
        "source_row": np.arange(12, dtype=np.int64),
        "transaction_id": np.arange(12, dtype=np.int32),
    })
    config = {
        "sampling": {"num_neighbors": [15, 2], "smoothing_alpha": 0.1},
        "audit": {"population_roots": 8, "max_fraud_roots": None, "repetitions": 2, "save_edge_level": True},
    }
    with tempfile.TemporaryDirectory() as temporary:
        output = Path(temporary)
        summary = audit_split(frame, config, output, "toy", 42)
        create_visualizations(output, sample_edges=100, seed=42)
        with np.load(output / "selected_edges.npz") as edges:
            edge_schema_valid = set(edges.files) >= {
                "method", "repetition", "channel", "root_tx", "candidate_tx", "counterpart_degree", "candidate_label"
            }
        with np.load(output / "local_jaccard.npz") as overlap:
            values = overlap["value"]
            overlap_valid = np.isfinite(values).all() and ((values >= 0) & (values <= 1)).all()
        degree_frame = pd.read_csv(output / "degree_distribution.csv")
        degree_contract_valid = (
            set(degree_frame["hop"]) == {1, 2}
            and {"p90", "p95", "p99", "degree_semantics"}.issubset(degree_frame.columns)
        )
        figures_valid = len(list((output / "figures").glob("*.png"))) == 4 and len(list((output / "figures").glob("*.pdf"))) == 4

    source = inspect.getsource(audit_split)
    checks = {
        "uniform_weights_are_one": np.allclose(candidate_score(counterpart_degree, "uniform"), 1.0),
        "topology_weights_decrease": bool(np.corrcoef(counterpart_degree, candidate_score(counterpart_degree, "topology"))[0, 1] < 0),
        "importance_weights_increase": bool(np.corrcoef(counterpart_degree, candidate_score(counterpart_degree, "importance"))[0, 1] > 0),
        "empirical_topology_selects_lower_degree": mean["topology"] < mean["uniform"],
        "empirical_importance_selects_higher_degree": mean["importance"] > mean["uniform"],
        "paired_streams_not_forced_identical": any(jaccard(first_selection["uniform"], first_selection[item]) < 1 for item in ("topology", "importance")),
        "csr_degree_correct": degree.tolist() == [3, 2, 1],
        "csr_candidates_correct": entity_candidates(order, offsets, 0).tolist() == [0, 2, 5],
        "sampling_function_has_no_labels": "label" not in inspect.signature(sample_without_replacement).parameters,
        "labels_read_after_sampling": source.index("sample_without_replacement") < source.index('edges["candidate_label"]'),
        "edge_level_contract_valid": edge_schema_valid,
        "jaccard_is_bounded": bool(overlap_valid),
        "degree_table_covers_hops_and_tail": bool(degree_contract_valid),
        "toy_audit_completed": summary["selected_edge_records"] > 0 and summary["local_jaccard_records"] > 0,
        "all_visualizations_created": figures_valid,
    }
    for name, passed in checks.items():
        print(f"[{'PASS' if passed else 'FAIL'}] {name}")
    write_json(Path(__file__).resolve().parent / "validation_report.json", {
        "status": "passed" if all(checks.values()) else "failed", "checks": checks,
        "empirical_selected_counterpart_degree_mean": mean,
    })
    if not all(checks.values()):
        raise SystemExit(1)


if __name__ == "__main__":
    main()


In [ ]:
import importlib.util
import subprocess
import sys
required = {'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib', 'yaml': 'PyYAML'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)


In [ ]:
!python validate_audit.py


In [ ]:
!python main.py --config config.kaggle.yaml --seed 42
